In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor


sys.path.append(str(Path.cwd().parent))

from src.data.features.build_features import add_lagged_returns, add_log_returns, add_log_rolling_volatility, add_rolling_z_score, add_rsi, load_prices_long
from src.data.targets.build_targets import add_volatility_target, add_direction_target
from src.models.evaluation import evaluate_predictions, walk_forward_folds, run_walk_forward, get_walk_forward_predictions
from src.backtest.backtest import run_backtest, compute_backtest_metrics
from src.backtest.position_sizing import compute_position_size


In [2]:
tickers = ['SPY','GOOG', 'NVDA', 'NFLX', 'MSFT']
df = load_prices_long("../data/raw_prices.parquet", tickers)
df.head()

Price,date,open,high,low,close,volume,ticker
0,2020-01-02,294.912997,296.143553,293.992353,296.125305,59151200,SPY
1,2020-01-03,292.743535,295.004113,292.688847,293.882935,77709700,SPY
2,2020-01-06,292.132782,295.086123,292.014280,295.004089,55653900,SPY
3,2020-01-07,294.438973,294.912981,293.727989,294.174652,40496400,SPY
4,2020-01-08,294.366059,296.954771,294.119959,295.742462,68296000,SPY


In [3]:
df = add_log_returns(df)
df = add_log_rolling_volatility(df, [5, 10, 20])
df = add_rsi(df, [14])
df = add_volatility_target(df, [5])

In [4]:
feature_cols = ['vol_5d', 'vol_10d', 'vol_20d', 'rsi_14d'] 
target_col = 'target_vol_5d'

In [ ]:
# Generates oos predictions across folds
pred_df = get_walk_forward_predictions(
    df=df, 
    feature_cols=feature_cols, 
    target_col=target_col, 
    model_factory=lambda: XGBRegressor(n_estimators=100, max_depth=3),
    train_size=252,
    test_size=21
)

In [6]:
# Scale predictions into positions
pred_df['position'] = compute_position_size(
    pred_df['predicted_log_vol'], target_vol=0.15, max_leverage=2.0
)

# Executes backtesting
backtest_results = run_backtest(
    df=pred_df, position_col='position', return_col='log_return', cost_bps=5.0
)

In [ ]:
# Computes metrics
strategy_metrics = compute_backtest_metrics(backtest_results['net_return'])
strategy_metrics

Sharpe          0.596090
Max_Drawdown   -0.614377
dtype: float64